In [1]:
import os
print(os.listdir('Results'))

['arima_7d_rebalance_test_summary.csv', 'arima_selected_orders.csv', 'ARIMA_test_7d_rebalance_actual_matrix.npy', 'ARIMA_test_7d_rebalance_forecast_matrix.npy', 'best_correlation_cholesky_LSTM.keras', 'dcc_garch_daily_covariance_matrices.npy', 'dcc_garch_daily_covariance_matrices_test.npy', 'Features and BO new return.npy', 'lstm_cov_matrices.npy', 'XGBoost Base Forecast new return.npy', 'XGBoost Features Forecast new return.npy']


In [2]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# PATHS
PRICES_DIR    = "klines csv data/prices_cleaned"
XGB_BASE_PATH = "Results New/XGBoost Base Forecast new return.npy"
XGB_FEAT_PATH = "Results New/XGBoost Features Forecast new return.npy"
XGB_BO_PATH   = "Results New/Features and BO new return.npy"
ARIMA_PATH    = "Results New/ARIMA_test_7d_rebalance_forecast_matrix.npy"
LSTM_PATH     = "Results New/lstm_cov_matrices.npy"
DCC_PATH      = "Results New/dcc_garch_daily_covariance_matrices_test.npy"
PERF_DIR      = "15 CMVO results new"

os.makedirs(PERF_DIR, exist_ok=True)

In [3]:
# 1. LOAD PRICE DATA
all_coins = sorted(os.listdir(PRICES_DIR))
all_coins = [c for c in all_coins if not c.endswith('.keras')
             and not c.endswith('.npy') and not c.endswith('.csv')]

glimpse  = pd.read_csv(PRICES_DIR + "/" + all_coins[0])
combined = pd.DataFrame({
    'date': pd.date_range(start='2022-04-01', periods=len(glimpse), freq='D')
})
for coin in all_coins:
    combined[coin] = pd.read_csv(PRICES_DIR + "/" + coin).sort_values(by='time')['close'].values

combined = combined.set_index('date')
print(f"Price data shape: {combined.shape}")
print(f"Coins ({len(all_coins)}): {all_coins}")
print(f"Date range: {combined.index[0].date()} to {combined.index[-1].date()}")

# 80/20 split — consistent with DCC and XGBoost
total_rows  = len(combined)
test_start  = int(total_rows * 0.8)
test_prices = combined.iloc[test_start:].copy()
print(f"\nTrain rows: {test_start}")
print(f"Test rows:  {len(test_prices)}")
print(f"Test period: {test_prices.index[0].date()} to {test_prices.index[-1].date()}")

Price data shape: (1371, 8)
Coins (8): ['ADAUSDT', 'BCHUSDT', 'BNBUSDT', 'BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'TRXUSDT', 'XRPUSDT']
Date range: 2022-04-01 to 2025-12-31

Train rows: 1096
Test rows:  275
Test period: 2025-04-01 to 2025-12-31


In [4]:
def parse_xgb(path, all_coins):
    xgb_raw = np.load(path, allow_pickle=True)
    df = pd.DataFrame(xgb_raw)

    if df.shape[1] == 1:
        df = df[0].astype(str).str.split(",", expand=True)

    df = df.iloc[:, 0:4].copy()
    df.columns = ['coin', 'actual', 'forecast', 'returns_predicted']
    df['coin'] = df['coin'].astype(str).str.strip()
    df = df[df['coin'].str.lower() != 'coin'].reset_index(drop=True)
    df['returns_predicted'] = pd.to_numeric(df['returns_predicted'], errors='coerce')

    mu_list = []
    for coin in all_coins:
        coin = str(coin).strip()
        coin_data = df[df['coin'] == coin]['returns_predicted'].values
        coin_data = coin_data[~np.isnan(coin_data)]
        mu_list.append(coin_data)

    non_empty = [x for x in mu_list if len(x) > 0]
    if len(non_empty) == 0:
        return np.array([])

    min_len = min(len(x) for x in non_empty)
    return np.column_stack([x[:min_len] for x in non_empty])

In [5]:
# 2. LOAD XGBOOST PREDICTIONS
print("Loading XGBoost Base predictions...")
mu_base = parse_xgb(XGB_BASE_PATH, all_coins)
print(f"mu_base shape: {mu_base.shape}  min: {mu_base.min():.4f}  max: {mu_base.max():.4f}")

print("\nLoading XGBoost Features predictions...")
mu_features = parse_xgb(XGB_FEAT_PATH, all_coins)
print(f"mu_features shape: {mu_features.shape}  min: {mu_features.min():.4f}  max: {mu_features.max():.4f}")

print("\nLoading XGBoost BO Features predictions...")
mu_bo = parse_xgb(XGB_BO_PATH, all_coins)
print(f"mu_bo shape: {mu_bo.shape}  min: {mu_bo.min():.4f}  max: {mu_bo.max():.4f}")


Loading XGBoost Base predictions...
mu_base shape: (39, 8)  min: -0.6855  max: 3.8824

Loading XGBoost Features predictions...
mu_features shape: (37, 8)  min: -1.5637  max: 1.6466

Loading XGBoost BO Features predictions...
mu_bo shape: (39, 8)  min: -0.4648  max: 0.4205


In [6]:
# 3. LOAD COVARIANCE MATRICES AND ALIGN

REBAL_DAYS = 7  # rebalance every 7 days on daily data

# ── LSTM ─────────────────────────────────────────────
print("Loading LSTM covariance matrices...")
lstm_full = np.load(LSTM_PATH, allow_pickle=True)
print(f"Full LSTM shape: {lstm_full.shape}")

lstm_test = lstm_full[test_start:]
lstm_cov  = lstm_test[::REBAL_DAYS]
print(f"LSTM test period shape: {lstm_test.shape}")
print(f"LSTM rebalance matrices: {lstm_cov.shape}")

# ── DCC ──────────────────────────────────────────────
print("\nLoading DCC-GARCH covariance matrices...")
dcc_full = np.load(DCC_PATH, allow_pickle=True)
dcc_cov  = dcc_full[::REBAL_DAYS]
print(f"DCC full shape: {dcc_full.shape}")
print(f"DCC rebalance matrices: {dcc_cov.shape}")

# ── ARIMA ─────────────────────────────────────────────
print("\nLoading ARIMA predictions...")
arima_mu = np.load(ARIMA_PATH)
print(f"arima_mu shape: {arima_mu.shape}  min: {arima_mu.min():.4f}  max: {arima_mu.max():.4f}")

# ── ALIGNMENT ─────────────────────────────────────────
test_rows       = len(test_prices)
expected_rebals = (test_rows - 1) // REBAL_DAYS
print(f"\nTest rows: {test_rows}, Expected rebalances: {expected_rebals}")

num_rebal = min(
    mu_base.shape[0],
    mu_features.shape[0],
    mu_bo.shape[0],
    lstm_cov.shape[0],
    dcc_cov.shape[0],
    expected_rebals,
)
print(f"Aligned rebalances: {num_rebal}")

mu_base     = mu_base[:num_rebal]
mu_features = mu_features[:num_rebal]
mu_bo       = mu_bo[:num_rebal]
lstm_cov    = lstm_cov[:num_rebal]
dcc_cov     = dcc_cov[:num_rebal]
arima_mu   = arima_mu[:num_rebal]

print(f"\nmu_base shape:     {mu_base.shape}")
print(f"mu_features shape: {mu_features.shape}")
print(f"mu_bo shape:       {mu_bo.shape}")
print(f"arima_mu shape:    {arima_mu.shape}")
print(f"lstm_cov shape:    {lstm_cov.shape}")
print(f"dcc_cov shape:     {dcc_cov.shape}")

Loading LSTM covariance matrices...
Full LSTM shape: (1340, 8, 8)
LSTM test period shape: (244, 8, 8)
LSTM rebalance matrices: (35, 8, 8)

Loading DCC-GARCH covariance matrices...
DCC full shape: (273, 8, 8)
DCC rebalance matrices: (39, 8, 8)

Loading ARIMA predictions...
arima_mu shape: (39, 8)  min: -0.0251  max: 0.0682

Test rows: 275, Expected rebalances: 39
Aligned rebalances: 35

mu_base shape:     (35, 8)
mu_features shape: (35, 8)
mu_bo shape:       (35, 8)
arima_mu shape:    (35, 8)
lstm_cov shape:    (35, 8, 8)
dcc_cov shape:     (35, 8, 8)


In [7]:
# 4. CMVO SOLVER
def solve_cmvo(mu, cov, risk_aversion=1.0):
    n       = len(mu)
    cov_reg = cov + 1e-8 * np.eye(n)

    mu_scale  = np.abs(mu).mean()
    cov_scale = np.diag(cov_reg).mean()
    scale     = mu_scale / cov_scale if cov_scale > 0 else 1.0
    cov_scaled = cov_reg * scale

    def neg_obj(w):
        return -(w @ mu - (risk_aversion / 2.0) * (w @ cov_scaled @ w))

    res = minimize(
        neg_obj,
        x0          = np.full(n, 1.0 / n),
        method      = 'SLSQP',
        bounds      = [(-0.1, 0.3)] * n,
        constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}],
        options     = {'ftol': 1e-9, 'maxiter': 1000},
    )

    if res.success:
        w  = np.clip(res.x, 0.0, 0.3)
        w /= w.sum()
        return w
    return np.full(n, 1.0 / n)

In [8]:
def run_portfolio(test_prices, all_coins, mu_array, cov_array,
                  risk_aversion=1.0, cost=None):
    # --- 1. SETUP ---
    cost       = cost if cost else 0.0
    prices     = test_prices[all_coins].values
    n          = len(all_coins)
    REBAL_DAYS = 7

    holdings         = (1.0 / n) / prices[0]
    portfolio_values = [1.0]
    pct_change       = [0.0]
    weight_history   = [(holdings * prices[0]) / 1.0]
    rebal_count      = 0

    # --- 2. SIMULATION LOOP ---
    for i in range(1, len(prices)):
        current_value = (holdings * prices[i]).sum()

        if i % REBAL_DAYS == 0:
            if rebal_count < len(mu_array):
                mu  = mu_array[rebal_count]
                cov = cov_array[rebal_count]
                # Optimizing weights based on XGBoost (mu) and LSTM (cov)
                new_weights = solve_cmvo(mu, cov, risk_aversion)
                rebal_count += 1
            else:
                new_weights = np.full(n, 1.0 / n)

            current_coin_values = holdings * prices[i]
            target_coin_values  = new_weights * current_value
            trade_amounts       = np.abs(current_coin_values - target_coin_values)
            total_tc            = (trade_amounts * cost).sum()

            net_value = current_value - total_tc
            holdings  = (new_weights * net_value) / prices[i]
            current_value = net_value

        # Track daily weight drift
        daily_weights = (holdings * prices[i]) / current_value
        weight_history.append(daily_weights)

        portfolio_values.append(current_value)
        pct_change.append((current_value / portfolio_values[-2]) - 1.0)

    # --- 3. CONSOLIDATE DATA ---
    df_perf = pd.DataFrame({
        'portfolio_values': portfolio_values,
        'pct_change': pct_change
    }, index=test_prices.index)

    df_weights = pd.DataFrame(weight_history, columns=all_coins, index=test_prices.index)

    final_df = pd.concat([df_perf, df_weights], axis=1)
    return final_df

# USAGE:
# results_df, metrics_dict = run_portfolio(test_prices, all_coins, mu_array, cov_array)

In [9]:
# 6. CALCULATE METRICS
def calculate_metrics(result):
    pct_changes      = result['pct_change'].iloc[1:]
    portfolio_values = result['portfolio_values']

    # cumulative and drawdown
    cumulative   = (1 + pct_changes).cumprod()
    rolling_peak = cumulative.cummax()
    drawdown     = (cumulative - rolling_peak) / rolling_peak
    mdd          = drawdown.min()

    # Sharpe
    mean   = pct_changes.mean()
    std    = pct_changes.std()
    sharpe = (mean / std) if std != 0 else 0.0

    # Calmar
    total_return = (portfolio_values.iloc[-1] / portfolio_values.iloc[0]) - 1
    calmar       = total_return / abs(mdd) if mdd != 0 else 0.0

    # Sortino
    downside = pct_changes[pct_changes < 0]
    down_std = downside.std()
    sortino  = (mean / down_std) if down_std != 0 else 0.0

    # Final value directly from portfolio values, matches Benchmarks
    final_value = portfolio_values.iloc[-1]

    return {
        'final_value':  final_value,
        'total_return': total_return,
        'mdd':          mdd,
        'sharpe':       sharpe,
        'sortino':      sortino,
        'calmar':       calmar,
    }

In [10]:
import numpy as np
import os

def save_metrics_npy(metrics_dict, full_path):
    """
    Creates necessary directories and saves a dictionary as a .npy file.

    Args:
        metrics_dict (dict): The dictionary containing your model results.
        full_path (str): The complete GDrive path including the filename.
    """
    # 1. Extract the directory portion of the path
    folder_path = os.path.dirname(full_path)

    # 2. Create folders if they don't exist
    if folder_path and not os.path.exists(folder_path):
        os.makedirs(folder_path, exist_ok=True)
        print(f"Created directory: {folder_path}")

    # 3. Save the file
    # Note: Metrics is saved as an object array to preserve the dictionary
    np.save(full_path, metrics_dict)
    print(f"Successfully saved metrics to: {full_path}")

In [11]:
# 7. RUN ALL MODELS AT BINANCE COST (0.1%)
costs = [0.001]
all_metrics = {}

for model_name, mu_array, cov in [
    ('xgb_base_lstm',     mu_base,     lstm_cov),
    ('xgb_features_lstm', mu_features, lstm_cov),
    ('xgb_bo_lstm',       mu_bo,       lstm_cov),
    ('xgb_base_dcc',      mu_base,     dcc_cov),
    ('xgb_features_dcc',  mu_features, dcc_cov),
    ('xgb_bo_dcc',        mu_bo,       dcc_cov),
    ('arima_lstm',        arima_mu,    lstm_cov),
    ('arima_dcc',         arima_mu,    dcc_cov),
]:
    res = run_portfolio(test_prices, all_coins, mu_array, cov, cost=0.001)
    m   = calculate_metrics(res)
    print(f"=== {model_name} (cost=0.001) ===")
    print(f"  Final value:  {m['final_value']:.4f}")
    print(f"  Total return: {m['total_return']:.4f}")
    print(f"  MDD:          {m['mdd']:.4f}")
    print(f"  Sharpe:       {m['sharpe']:.4f}")
    print(f"  Sortino:      {m['sortino']:.4f}")
    print(f"  Calmar:       {m['calmar']:.4f}")
    print()
    all_metrics[model_name] = m
    res.to_csv(os.path.join(PERF_DIR, f'{model_name}.csv'), index=False)

save_metrics_npy(all_metrics, os.path.join(PERF_DIR, 'all_portfolio_metrics.npy'))
print("Saved all_portfolio_metrics.npy")

=== xgb_base_lstm (cost=0.001) ===
  Final value:  1.1590
  Total return: 0.1590
  MDD:          -0.3487
  Sharpe:       0.0333
  Sortino:      0.0479
  Calmar:       0.4560

=== xgb_features_lstm (cost=0.001) ===
  Final value:  1.0391
  Total return: 0.0391
  MDD:          -0.3117
  Sharpe:       0.0185
  Sortino:      0.0295
  Calmar:       0.1254

=== xgb_bo_lstm (cost=0.001) ===
  Final value:  1.0960
  Total return: 0.0960
  MDD:          -0.4088
  Sharpe:       0.0263
  Sortino:      0.0399
  Calmar:       0.2347

=== xgb_base_dcc (cost=0.001) ===
  Final value:  1.1229
  Total return: 0.1229
  MDD:          -0.3609
  Sharpe:       0.0292
  Sortino:      0.0418
  Calmar:       0.3404

=== xgb_features_dcc (cost=0.001) ===
  Final value:  1.0536
  Total return: 0.0536
  MDD:          -0.2864
  Sharpe:       0.0203
  Sortino:      0.0325
  Calmar:       0.1871

=== xgb_bo_dcc (cost=0.001) ===
  Final value:  1.0495
  Total return: 0.0495
  MDD:          -0.4242
  Sharpe:       0.0

In [12]:
# 8. BENCHMARK COMPARISON — computed on same test period as CMVO
# 1/N Equal Weight (buy and hold, no rebalancing, no cost)
def run_equal_weight(test_prices, all_coins):
    n      = len(all_coins)
    prices = test_prices[all_coins].values
    holdings = (1.0 / n) / prices[0]

    portfolio_values = [1.0]
    pct_change       = [0.0]

    for i in range(1, len(prices)):
        current_value = (holdings * prices[i]).sum()
        portfolio_values.append(current_value)
        pct_change.append((current_value / portfolio_values[-2]) - 1.0)

    return pd.DataFrame({
        'portfolio_values': portfolio_values,
        'pct_change':       pct_change,
    })

# Statistical MV (historical mean/cov, no ML forecasting)
def run_mv_benchmark(test_prices, all_coins, lookback=60,
                     risk_aversion=1.0, cost=0.001):
    n      = len(all_coins)
    prices = test_prices[all_coins].values

    holdings         = (1.0 / n) / prices[0]
    portfolio_values = [1.0]
    pct_change       = [0.0]
    rebal_count      = 0
    REBAL_DAYS       = 7

    for i in range(1, len(prices)):
        current_value = (holdings * prices[i]).sum()

        if i % REBAL_DAYS == 0:
            start = max(0, i - lookback)
            window_prices  = prices[start:i]
            window_returns = np.diff(window_prices, axis=0) / window_prices[:-1]

            if len(window_returns) >= 2:
                mu_hist  = window_returns.mean(axis=0)
                cov_hist = np.cov(window_returns.T) + 1e-8 * np.eye(n)
                new_weights = solve_cmvo(mu_hist, cov_hist, risk_aversion)
            else:
                new_weights = np.full(n, 1.0 / n)

            current_coin_values = holdings * prices[i]
            target_coin_values  = new_weights * current_value
            trade_amounts       = np.abs(current_coin_values - target_coin_values)
            total_tc            = (trade_amounts * cost).sum()

            net_value     = current_value - total_tc
            holdings      = (new_weights * net_value) / prices[i]
            current_value = net_value

        portfolio_values.append(current_value)
        pct_change.append((current_value / portfolio_values[-2]) - 1.0)

    return pd.DataFrame({
        'portfolio_values': portfolio_values,
        'pct_change':       pct_change,
    })

# Run benchmarks
res_1n = run_equal_weight(test_prices, all_coins)
m_1n   = calculate_metrics(res_1n)

res_mv = run_mv_benchmark(test_prices, all_coins, cost=0.001)
m_mv   = calculate_metrics(res_mv)

# Print full comparison
print("=== BENCHMARK COMPARISON (test period: Apr–Dec 2025) ===")
print(f"{'Model':<25} {'Final':>8} {'Sharpe':>8} {'MDD':>8} {'Sortino':>8} {'Calmar':>8}")
print("-" * 70)
print(f"{'1/N Equal Weight':<25} {m_1n['final_value']:>8.4f} {m_1n['sharpe']:>8.4f} {m_1n['mdd']:>8.4f} {m_1n['sortino']:>8.4f} {m_1n['calmar']:>8.4f}")
print(f"{'Statistical MV':<25} {m_mv['final_value']:>8.4f} {m_mv['sharpe']:>8.4f} {m_mv['mdd']:>8.4f} {m_mv['sortino']:>8.4f} {m_mv['calmar']:>8.4f}")
print()
for model_name, m in all_metrics.items():
    print(f"{model_name:<25} {m['final_value']:>8.4f} {m['sharpe']:>8.4f} {m['mdd']:>8.4f} {m['sortino']:>8.4f} {m['calmar']:>8.4f}")

# Save benchmark results
all_metrics["1/N Equal Weight"] = m_1n
all_metrics["Statistical MV"] = m_mv
save_metrics_npy(all_metrics, os.path.join(PERF_DIR, 'all_portfolio_metrics.npy'))

res_1n.to_csv(os.path.join(PERF_DIR, 'benchmark_1n.csv'), index=False)
res_mv.to_csv(os.path.join(PERF_DIR, 'benchmark_mv.csv'), index=False)



=== BENCHMARK COMPARISON (test period: Apr–Dec 2025) ===
Model                        Final   Sharpe      MDD  Sortino   Calmar
----------------------------------------------------------------------
1/N Equal Weight            1.1855   0.0363  -0.3377   0.0545   0.5491
Statistical MV              1.2305   0.0420  -0.3234   0.0608   0.7128

xgb_base_lstm               1.1590   0.0333  -0.3487   0.0479   0.4560
xgb_features_lstm           1.0391   0.0185  -0.3117   0.0295   0.1254
xgb_bo_lstm                 1.0960   0.0263  -0.4088   0.0399   0.2347
xgb_base_dcc                1.1229   0.0292  -0.3609   0.0418   0.3404
xgb_features_dcc            1.0536   0.0203  -0.2864   0.0325   0.1871
xgb_bo_dcc                  1.0495   0.0211  -0.4242   0.0321   0.1167
arima_lstm                  0.9161  -0.0002  -0.3867  -0.0003  -0.2169
arima_dcc                   0.9738   0.0088  -0.3632   0.0130  -0.0722
Successfully saved metrics to: 15 CMVO results new\all_portfolio_metrics.npy


In [13]:
# 8. BENCHMARK COMPARISON — computed on same test period as CMVO
# 1/N Equal Weight (buy and hold, no rebalancing, no cost)
def run_equal_weight(test_prices, all_coins):
    n      = len(all_coins)
    prices = test_prices[all_coins].values
    holdings = (1.0 / n) / prices[0]

    portfolio_values = [1.0]
    pct_change       = [0.0]

    for i in range(1, len(prices)):
        current_value = (holdings * prices[i]).sum()
        portfolio_values.append(current_value)
        pct_change.append((current_value / portfolio_values[-2]) - 1.0)

    return pd.DataFrame({
        'portfolio_values': portfolio_values,
        'pct_change':       pct_change,
    })

# Statistical MV (historical mean/cov, no ML forecasting)
def run_mv_benchmark(test_prices, all_coins, lookback=60,
                     risk_aversion=1.0, cost=0.001):
    n = len(all_coins)
    prices = test_prices[all_coins].values
    dates = test_prices.index  # Keep track of dates for the final DataFrame

    # Initialize weights: start with Equal Weight (1/n)
    current_weights = np.full(n, 1.0 / n)
    holdings = current_weights / prices[0]

    portfolio_values = [1.0]
    pct_change = [0.0]
    # List to store weight arrays for every single day
    weight_history = [current_weights.copy()]

    REBAL_DAYS = 7

    for i in range(1, len(prices)):
        # Calculate current value before rebalancing
        current_value = (holdings * prices[i]).sum()

        if i % REBAL_DAYS == 0:
            start = max(0, i - lookback)
            window_prices = prices[start:i]
            window_returns = np.diff(window_prices, axis=0) / window_prices[:-1]

            if len(window_returns) >= 2:
                mu_hist = window_returns.mean(axis=0)
                cov_hist = np.cov(window_returns.T) + 1e-8 * np.eye(n)
                # Solve for new weights
                current_weights = solve_cmvo(mu_hist, cov_hist, risk_aversion)
            else:
                current_weights = np.full(n, 1.0 / n)

            # Calculate Transaction Costs
            current_coin_values = holdings * prices[i]
            target_coin_values = current_weights * current_value
            trade_amounts = np.abs(current_coin_values - target_coin_values)
            total_tc = (trade_amounts * cost).sum()

            # Adjust holdings based on net value after costs
            net_value = current_value - total_tc
            holdings = (current_weights * net_value) / prices[i]
            current_value = net_value
        else:
            # On non-rebalance days, weights shift naturally with price movements
            # We calculate the "drifting" weights
            current_coin_values = holdings * prices[i]
            current_weights = current_coin_values / current_value

        portfolio_values.append(current_value)
        pct_change.append((current_value / portfolio_values[-2]) - 1.0)
        weight_history.append(current_weights.copy())

    # Create the base DataFrame
    results_df = pd.DataFrame({
        'date': dates,
        'portfolio_values': portfolio_values,
        'pct_change': pct_change,
    })

    # Add weight columns: e.g., 'weight_BTC', 'weight_ETH', etc.
    weights_df = pd.DataFrame(weight_history, columns=[f'{c}' for c in all_coins])

    return pd.concat([results_df, weights_df], axis=1).set_index('date')

# Run benchmarks
res_1n = run_equal_weight(test_prices, all_coins)
m_1n   = calculate_metrics(res_1n)

res_mv = run_mv_benchmark(test_prices, all_coins, cost=0.001)
m_mv   = calculate_metrics(res_mv)
print(res_mv.head())

# Print full comparison
print("=== BENCHMARK COMPARISON (test period: Apr–Dec 2025) ===")
print(f"{'Model':<25} {'Final':>8} {'Sharpe':>8} {'MDD':>8} {'Sortino':>8} {'Calmar':>8}")
print("-" * 70)
print(f"{'1/N Equal Weight':<25} {m_1n['final_value']:>8.4f} {m_1n['sharpe']:>8.4f} {m_1n['mdd']:>8.4f} {m_1n['sortino']:>8.4f} {m_1n['calmar']:>8.4f}")
print(f"{'Statistical MV':<25} {m_mv['final_value']:>8.4f} {m_mv['sharpe']:>8.4f} {m_mv['mdd']:>8.4f} {m_mv['sortino']:>8.4f} {m_mv['calmar']:>8.4f}")
print()
for model_name, m in all_metrics.items():
    print(f"{model_name:<25} {m['final_value']:>8.4f} {m['sharpe']:>8.4f} {m['mdd']:>8.4f} {m['sortino']:>8.4f} {m['calmar']:>8.4f}")

# Save benchmark results
all_metrics["1/N Equal Weight"] = m_1n
all_metrics["Statistical MV"] = m_mv
save_metrics_npy(all_metrics, os.path.join(PERF_DIR, 'all_portfolio_metrics.npy'))

res_1n.to_csv(os.path.join(PERF_DIR, 'benchmark_1n.csv'), index=False)
res_mv.to_csv(os.path.join(PERF_DIR, 'benchmark_mv.csv'), index=False)

            portfolio_values  pct_change   ADAUSDT   BCHUSDT   BNBUSDT  \
date                                                                     
2025-04-01          1.000000    0.000000  0.125000  0.125000  0.125000   
2025-04-02          0.953985   -0.046015  0.123868  0.124608  0.126645   
2025-04-03          0.966531    0.013151  0.124323  0.126474  0.125422   
2025-04-04          0.980115    0.014055  0.124390  0.124763  0.124735   
2025-04-05          0.975603   -0.004604  0.123906  0.126462  0.124352   

             BTCUSDT   ETHUSDT   SOLUSDT   TRXUSDT   XRPUSDT  
date                                                          
2025-04-01  0.125000  0.125000  0.125000  0.125000  0.125000  
2025-04-02  0.126964  0.123480  0.121461  0.128991  0.123984  
2025-04-03  0.126374  0.123371  0.119629  0.129546  0.124860  
2025-04-04  0.125636  0.121637  0.123630  0.128180  0.127029  
2025-04-05  0.125688  0.121469  0.121703  0.127964  0.128455  
=== BENCHMARK COMPARISON (test period: A

In [14]:
# 9. FULL COMPARISON TABLE
print("=== FULL RESULTS (test period: Apr–Dec 2025, cost=0.001) ===")
print(f"{'Model':<25} {'Final':>8} {'Return':>8} {'Sharpe':>8} {'MDD':>8} {'Sortino':>8} {'Calmar':>8}")
print("-" * 80)

for name, m in [('1/N Equal Weight', m_1n), ('Statistical MV', m_mv)]:
    print(f"{name:<25} {m['final_value']:>8.4f} {m['total_return']:>8.4f} {m['sharpe']:>8.4f} {m['mdd']:>8.4f} {m['sortino']:>8.4f} {m['calmar']:>8.4f}")

print("-" * 80)

for model_name, m in all_metrics.items():
    print(f"{model_name:<25} {m['final_value']:>8.4f} {m['total_return']:>8.4f} {m['sharpe']:>8.4f} {m['mdd']:>8.4f} {m['sortino']:>8.4f} {m['calmar']:>8.4f}")

=== FULL RESULTS (test period: Apr–Dec 2025, cost=0.001) ===
Model                        Final   Return   Sharpe      MDD  Sortino   Calmar
--------------------------------------------------------------------------------
1/N Equal Weight            1.1855   0.1855   0.0363  -0.3377   0.0545   0.5491
Statistical MV              1.2305   0.2305   0.0420  -0.3234   0.0608   0.7128
--------------------------------------------------------------------------------
xgb_base_lstm               1.1590   0.1590   0.0333  -0.3487   0.0479   0.4560
xgb_features_lstm           1.0391   0.0391   0.0185  -0.3117   0.0295   0.1254
xgb_bo_lstm                 1.0960   0.0960   0.0263  -0.4088   0.0399   0.2347
xgb_base_dcc                1.1229   0.1229   0.0292  -0.3609   0.0418   0.3404
xgb_features_dcc            1.0536   0.0536   0.0203  -0.2864   0.0325   0.1871
xgb_bo_dcc                  1.0495   0.0495   0.0211  -0.4242   0.0321   0.1167
arima_lstm                  0.9161  -0.0839  -0.0002  -0.

# TESTING FOR SHORTING

In [15]:
# TEST OPTIMAL LONG AND SHORT BOUNDS (xgb_bo_lstm, cost=0.001)

long_bounds  = [0.20, 0.25, 0.30, 0.40, 0.50]
short_bounds = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30]

print("=== BOUND SEARCH (xgb_bo_lstm, cost=0.001) ===")
print(f"{'Long':>6} {'Short':>6} {'Final':>8} {'Sharpe':>8} {'MDD':>8} {'Calmar':>8}")
print("-" * 50)

best_sharpe = -np.inf
best_long   = None
best_short  = None

for long_b in long_bounds:
    for short_b in short_bounds:

        # feasibility check: n * long_b >= 1.0
        if len(all_coins) * long_b < 1.0:
            print(f"{long_b:>6.2f} {short_b:>6.2f} {'INFEASIBLE':>8}")
            continue

        def solve_test(mu, cov, lb=long_b, sb=short_b):
            n          = len(mu)
            cov_reg    = cov + 1e-8 * np.eye(n)
            mu_scale   = np.abs(mu).mean()
            cov_scale  = np.diag(cov_reg).mean()
            scale      = mu_scale / cov_scale if cov_scale > 0 else 1.0
            cov_scaled = cov_reg * scale
            def neg_obj(w):
                return -(w @ mu - (1.0 / 2.0) * (w @ cov_scaled @ w))
            res = minimize(
                neg_obj,
                x0          = np.full(n, 1.0 / n),
                method      = 'SLSQP',
                bounds      = [(-sb, lb)] * n,
                constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}],
                options     = {'ftol': 1e-9, 'maxiter': 1000},
            )
            if res.success:
                w = np.clip(res.x, -sb, lb)
                w /= w.sum()
                return w
            return np.full(n, 1.0 / n)

        prices      = test_prices[all_coins].values
        n           = len(all_coins)
        holdings    = (1.0 / n) / prices[0]
        pv, pc      = [1.0], [0.0]
        rebal_count = 0

        for i in range(1, len(prices)):
            current_value = (holdings * prices[i]).sum()
            if i % 7 == 0:
                if rebal_count < len(mu_bo):
                    new_weights = solve_test(mu_bo[rebal_count], lstm_cov[rebal_count])
                    rebal_count += 1
                else:
                    new_weights = np.full(n, 1.0 / n)
                tc            = (np.abs(holdings * prices[i] - new_weights * current_value) * 0.001).sum()
                net_value     = current_value - tc
                holdings      = (new_weights * net_value) / prices[i]
                current_value = net_value
            pv.append(current_value)
            pc.append((current_value / pv[-2]) - 1.0)

        res_df = pd.DataFrame({'portfolio_values': pv, 'pct_change': pc})
        m      = calculate_metrics(res_df)

        flag = ' ← best' if m['sharpe'] > best_sharpe and m['mdd'] > -1.0 else ''
        if m['sharpe'] > best_sharpe and m['mdd'] > -1.0:
            best_sharpe = m['sharpe']
            best_long   = long_b
            best_short  = short_b

        mdd_flag = ' ⚠' if m['mdd'] < -1.0 else ''
        print(f"{long_b:>6.2f} {short_b:>6.2f} {m['final_value']:>8.4f} "
              f"{m['sharpe']:>8.4f} {m['mdd']:>8.4f}{mdd_flag} {m['calmar']:>8.4f}{flag}")

print()
print(f"Best: long={best_long}, short={best_short}, Sharpe={best_sharpe:.4f}")
print(f"Note: ⚠ means MDD < -1.0 (portfolio went negative — unrealistic, excluded)")

=== BOUND SEARCH (xgb_bo_lstm, cost=0.001) ===
  Long  Short    Final   Sharpe      MDD   Calmar
--------------------------------------------------
  0.20   0.00   1.1071   0.0274  -0.4022   0.2663 ← best
  0.20   0.05   1.1288   0.0298  -0.3941   0.3268 ← best
  0.20   0.10   1.1306   0.0300  -0.3995   0.3269 ← best
  0.20   0.15   1.1190   0.0288  -0.3985   0.2987
  0.20   0.20   1.1060   0.0276  -0.3993   0.2655
  0.20   0.30   1.0578   0.0227  -0.4073   0.1419
  0.25   0.00   1.0767   0.0242  -0.4173   0.1838
  0.25   0.05   1.0852   0.0254  -0.4304   0.1979
  0.25   0.10   1.0807   0.0251  -0.4384   0.1840
  0.25   0.15   1.0976   0.0270  -0.4340   0.2248
  0.25   0.20   1.0942   0.0269  -0.4431   0.2127
  0.25   0.30   1.0683   0.0248  -0.4525   0.1510
  0.30   0.00   1.0243   0.0186  -0.4456   0.0545
  0.30   0.05   1.0372   0.0205  -0.4460   0.0834
  0.30   0.10   1.0575   0.0230  -0.4582   0.1254
  0.30   0.15   1.0488   0.0225  -0.4732   0.1032
  0.30   0.20   1.0475   0.0228

In [16]:
import pandas as pd
import os

# Convert the numpy array to a DataFrame
# If you have your ticker names, replace ['Asset1', 'Asset2'...] with your list
mu_df = pd.DataFrame(mu_bo)
print(mu_df.head())

clean_coins = [c for c in all_coins if not c.endswith(('.keras', '.npy', '.csv'))]
mu_df.columns = clean_coins
print(mu_df.head())

n_periods, n_assets, _ = lstm_cov.shape

print(f"Min Correlation in tensor: {lstm_cov.min()}")
print(f"Max Correlation in tensor: {lstm_cov.max()}")

# 2. Identify the off-diagonal indices
iu = np.triu_indices(n_assets, k=1) # Indices for the upper triangle
il = np.tril_indices(n_assets, k=-1) # Indices for the lower triangle

# 3. Calculate mean of off-diagonal elements for each slice
# We combine upper and lower triangle indices to get all non-diagonal correlations
avg_corr_per_period = np.array([
    matrix[np.concatenate((iu[0], il[0])), np.concatenate((iu[1], il[1]))].mean()
    for matrix in lstm_cov
])
avg_cov= pd.DataFrame(avg_corr_per_period)

# Now 'avg_corr_per_period' is a (35,) array
print(f"Average Correlation for first 3 periods: {avg_corr_per_period[:3]}")

# Create the directory if it doesn't exist
os.makedirs("15 CMVO results/bestmodelpredictions", exist_ok=True)

os.makedirs("15 CMVO results/bestmodelpredictions", exist_ok=True)

# Now save it
mu_df.to_csv(os.path.join("15 CMVO results/bestmodelpredictions", 'xgb_bo_returns.csv'), index=False)
avg_cov.to_csv(os.path.join("15 CMVO results/bestmodelpredictions", 'lstm_correlations.csv'), index=False)

          0         1         2         3         4         5         6  \
0  0.028126  0.035386  0.043554  0.005043  0.025516  0.143629 -0.014230   
1 -0.045823 -0.139451  0.068411  0.083760 -0.039686  0.038688 -0.006519   
2 -0.008363  0.031727 -0.025695  0.027371 -0.011437  0.015976 -0.102474   
3  0.005420  0.014635  0.028724 -0.027540  0.037044  0.238774 -0.062346   
4  0.044197 -0.032752 -0.011656  0.012238 -0.038389  0.034489 -0.068460   

          7  
0  0.067425  
1  0.244487  
2  0.182206  
3  0.291320  
4  0.178130  
    ADAUSDT   BCHUSDT   BNBUSDT   BTCUSDT   ETHUSDT   SOLUSDT   TRXUSDT  \
0  0.028126  0.035386  0.043554  0.005043  0.025516  0.143629 -0.014230   
1 -0.045823 -0.139451  0.068411  0.083760 -0.039686  0.038688 -0.006519   
2 -0.008363  0.031727 -0.025695  0.027371 -0.011437  0.015976 -0.102474   
3  0.005420  0.014635  0.028724 -0.027540  0.037044  0.238774 -0.062346   
4  0.044197 -0.032752 -0.011656  0.012238 -0.038389  0.034489 -0.068460   

    XRPUSDT  
